# Step 1 — Recommended implementation

This is the corrected-output-folder version recommended for new analyses.


In [ ]:
"""
Integrated nuclear-export pipeline with selectable 2D / 3D r_norm.

Logic:
  1) Load transcripts
  2) Keep nuclear transcripts
  3) FIRST sample cells
  4) Compute 2D r_norm
  5) If 3D mode:
       - estimate nuclear diameters
       - Wicksell correction
       - compute 3D r_norm
  6) Set selected r_norm:
       2D -> r_norm = r_norm_2d
       3D -> r_norm = r_norm_3d
  7) Fit per-cell PDE using selected r_norm
  8) Optional spatial KS map
  9) Compute export probabilities using selected r_norm
 10) Build delta, median-standardized delta, cytoplasm and future matrices

Output structure:

pipeline_out/
├── mode_2D/
│   ├── out_part1/
│   ├── out_part2/
│   ├── out_part3/
│   └── descriptive/
│
└── mode_3D/
    ├── out_part1/
    ├── out_part2/
    ├── out_part3/
    └── descriptive/
"""

import os
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from scipy.optimize import minimize
from scipy.stats import (
    kstest,
    gaussian_kde,
    pearsonr,
    spearmanr,
    linregress,
)
from scipy.special import i0, i1
from scipy.interpolate import interp1d
from scipy.ndimage import gaussian_filter1d
from tqdm import tqdm


# ============================================================
# PATHS
# ============================================================

_CWD_DATA_DIR = os.path.join(os.getcwd(), "data", "test_data")
_DEFAULT_WORKDIR = (
    _CWD_DATA_DIR
    if os.path.isdir(_CWD_DATA_DIR)
    else os.path.join("..", "data", "test_data")
)
BASE_DIR = os.path.abspath(
    os.environ.get("SPATIAL_VELO_WORKDIR", _DEFAULT_WORKDIR)
)

TX_IN_FILE = os.path.join(
    BASE_DIR,
    "xenium_test_tx_file.csv"
)

META_FILE = os.path.join(
    BASE_DIR,
    "xenium_test_metadata.csv"
)


# ============================================================
# PARAMETERS
# ============================================================

# ------------------------------------------------------------
# Select normalization mode here:
# "2D" or "3D"
# ------------------------------------------------------------
NORM_MODE = os.environ.get("SPATIAL_VELO_NORM_MODE", "2D").upper()


# Sampling
SAMPLE_SIZE = 10000
RANDOM_STATE = 42

np.random.seed(RANDOM_STATE)


# Transcript columns
TX_CELL_COL = "cell"
TX_GENE_COL = "gene"
TX_COMP_COL = "CellComp"

TX_X_COL = "x"
TX_Y_COL = "y"


# Metadata columns
META_CELL_COL = "cell"

META_X_COL = "global_x"
META_Y_COL = "global_y"


# Fallback transcript columns
TX_CELL_FALLBACK = "cell_id"
TX_GENE_FALLBACK = "target"

TX_X_FALLBACK = "global_x"
TX_Y_FALLBACK = "global_y"


# Compartment labels
NUCLEAR_LABEL = "nuclear"
CYTOPLASM_LABEL = "cytoplasm"


# PDE
MIN_TX_PER_CELL = 50


# Plot color limits
PLOT_VMIN_PERCENTILE = 5
PLOT_VMAX_PERCENTILE = 95


# 3D correction
MAX_DIAMETER = None

DIAM_CELL_COL = "cell"
DIAMETER_COL = "diameter_px"


# ============================================================
# CHECK MODE
# ============================================================

NORM_MODE = NORM_MODE.upper().strip()

if NORM_MODE not in {"2D", "3D"}:
    raise ValueError(
        'NORM_MODE must be either "2D" or "3D".'
    )


# ============================================================
# OUTPUT PATHS
# ============================================================

OUT_ROOT = os.path.join(
    BASE_DIR,
    "pipeline_out"
)

OUT_DIR = os.path.join(
    OUT_ROOT,
    f"mode_{NORM_MODE}"
)

OUT_PART1 = os.path.join(
    OUT_DIR,
    "out_part1"
)

OUT_PART2 = os.path.join(
    OUT_DIR,
    "out_part2"
)

OUT_PART3 = os.path.join(
    OUT_DIR,
    "out_part3"
)

OUT_DESC = os.path.join(
    OUT_DIR,
    "descriptive"
)


for p in [
    OUT_PART1,
    OUT_PART2,
    OUT_PART3,
    OUT_DESC,
]:
    os.makedirs(
        p,
        exist_ok=True
    )


# ============================================================
# OUTPUT FILES
# ============================================================

TX_RNORM_OUT = os.path.join(
    OUT_PART1,
    "tx_nuclear_with_rnorm.csv"
)

FIT_OUT = os.path.join(
    OUT_PART2,
    "fit_results_all_cells.csv"
)

DELTA_OUT_CSV = os.path.join(
    OUT_PART3,
    "delta_export_prob.csv"
)

DELTA_MEDIAN_OUT = os.path.join(
    OUT_PART3,
    "delta_export_prob_median.csv"
)

CYTO_OUT_MATRIX = os.path.join(
    OUT_PART3,
    "transcripts_cytoplasm_matrix_common.csv"
)

FUTURE_OUT = os.path.join(
    OUT_PART3,
    "future.csv"
)

DIAMETER_CSV_OUT = os.path.join(
    OUT_DESC,
    "nuclear_diameters.csv"
)


# ============================================================
# GENERAL FUNCTIONS
# ============================================================

def standardize_transcripts(
    tx_df: pd.DataFrame,
    require_gene: bool = False
) -> pd.DataFrame:

    """
    Standardize transcript columns to:

        cell
        gene
        CellComp
        x
        y
    """

    tx_df = tx_df.copy()

    # --------------------------------------------------------
    # Cell
    # --------------------------------------------------------

    if "cell" not in tx_df.columns:

        if TX_CELL_COL in tx_df.columns:
            tx_df["cell"] = tx_df[TX_CELL_COL]

        elif TX_CELL_FALLBACK in tx_df.columns:
            tx_df["cell"] = tx_df[TX_CELL_FALLBACK]

        else:
            raise RuntimeError(
                "Input transcripts file missing cell column. "
                f"Tried: cell, {TX_CELL_COL}, "
                f"{TX_CELL_FALLBACK}"
            )

    # --------------------------------------------------------
    # Cell compartment
    # --------------------------------------------------------

    if "CellComp" not in tx_df.columns:

        if TX_COMP_COL in tx_df.columns:
            tx_df["CellComp"] = tx_df[TX_COMP_COL]

        else:
            raise RuntimeError(
                "Input transcripts file missing "
                "CellComp column."
            )

    # --------------------------------------------------------
    # Gene
    # --------------------------------------------------------

    if "gene" not in tx_df.columns:

        if TX_GENE_COL in tx_df.columns:
            tx_df["gene"] = tx_df[TX_GENE_COL]

        elif TX_GENE_FALLBACK in tx_df.columns:
            tx_df["gene"] = tx_df[TX_GENE_FALLBACK]

        elif require_gene:
            raise RuntimeError(
                "Input transcripts file missing gene column."
            )

    # --------------------------------------------------------
    # X
    # --------------------------------------------------------

    if "x" not in tx_df.columns:

        if TX_X_COL in tx_df.columns:
            tx_df["x"] = tx_df[TX_X_COL]

        elif TX_X_FALLBACK in tx_df.columns:
            tx_df["x"] = tx_df[TX_X_FALLBACK]

        else:
            raise RuntimeError(
                "Input transcripts file missing x coordinate."
            )

    # --------------------------------------------------------
    # Y
    # --------------------------------------------------------

    if "y" not in tx_df.columns:

        if TX_Y_COL in tx_df.columns:
            tx_df["y"] = tx_df[TX_Y_COL]

        elif TX_Y_FALLBACK in tx_df.columns:
            tx_df["y"] = tx_df[TX_Y_FALLBACK]

        else:
            raise RuntimeError(
                "Input transcripts file missing y coordinate."
            )

    # --------------------------------------------------------
    # Types
    # --------------------------------------------------------

    tx_df["cell"] = tx_df["cell"].astype(str)

    tx_df["CellComp"] = (
        tx_df["CellComp"]
        .astype(str)
    )

    if "gene" in tx_df.columns:
        tx_df["gene"] = (
            tx_df["gene"]
            .astype(str)
        )

    tx_df["x"] = pd.to_numeric(
        tx_df["x"],
        errors="coerce"
    )

    tx_df["y"] = pd.to_numeric(
        tx_df["y"],
        errors="coerce"
    )

    tx_df = (
        tx_df
        .dropna(
            subset=["x", "y"]
        )
        .copy()
    )

    return tx_df


def standardize_metadata(
    meta_df: pd.DataFrame
) -> pd.DataFrame:

    """
    Convert metadata coordinates to:

        cell
        meta_x
        meta_y
    """

    required = [
        META_CELL_COL,
        META_X_COL,
        META_Y_COL,
    ]

    for col in required:

        if col not in meta_df.columns:

            raise RuntimeError(
                f"Metadata file missing required "
                f"column: {col}"
            )

    meta_df = meta_df.rename(
        columns={
            META_CELL_COL: "cell",
            META_X_COL: "meta_x",
            META_Y_COL: "meta_y",
        }
    ).copy()

    meta_df["cell"] = (
        meta_df["cell"]
        .astype(str)
    )

    meta_df["meta_x"] = pd.to_numeric(
        meta_df["meta_x"],
        errors="coerce"
    )

    meta_df["meta_y"] = pd.to_numeric(
        meta_df["meta_y"],
        errors="coerce"
    )

    meta_df = meta_df.dropna(
        subset=[
            "meta_x",
            "meta_y",
        ]
    ).copy()

    return meta_df


def get_percentile_limits(
    values,
    low_pct=5,
    high_pct=95
):

    arr = (
        pd.to_numeric(
            pd.Series(values),
            errors="coerce"
        )
        .dropna()
        .to_numpy(dtype=float)
    )

    if len(arr) == 0:
        return 0.0, 1.0

    vmin = float(
        np.percentile(
            arr,
            low_pct
        )
    )

    vmax = float(
        np.percentile(
            arr,
            high_pct
        )
    )

    if np.isclose(vmin, vmax):
        vmax = vmin + 1e-12

    return vmin, vmax


# ============================================================
# STEP 0
# LOAD TRANSCRIPTS
# ============================================================

def load_transcripts(
    tx_file: str
) -> pd.DataFrame:

    print(
        "\n"
        "============================================================"
    )

    print(
        f"Running pipeline in {NORM_MODE} mode"
    )

    print(
        "============================================================"
    )

    print(
        "\nStep 0: Loading transcripts..."
    )

    if not os.path.exists(tx_file):

        raise FileNotFoundError(
            f"Transcript file not found:\n{tx_file}"
        )

    tx_df = pd.read_csv(
        tx_file,
        low_memory=False
    )

    print(
        f"  Loaded transcript rows: "
        f"{len(tx_df):,}"
    )

    tx_df = standardize_transcripts(
        tx_df,
        require_gene=False
    )

    return tx_df


def get_nuclear_transcripts(
    tx_df: pd.DataFrame
) -> pd.DataFrame:

    nuclear_tx = tx_df[
        tx_df["CellComp"]
        .astype(str)
        .str.lower()
        ==
        NUCLEAR_LABEL.lower()
    ].copy()

    if len(nuclear_tx) == 0:

        raise RuntimeError(
            "No nuclear transcripts found."
        )

    print(
        f"  Nuclear transcripts: "
        f"{len(nuclear_tx):,}"
    )

    print(
        f"  Cells with nuclear transcripts: "
        f"{nuclear_tx['cell'].nunique():,}"
    )

    return nuclear_tx


def sample_cells_first(
    nuclear_tx: pd.DataFrame,
    sample_size: int,
    random_state: int = 42
) -> pd.DataFrame:

    """
    Sample cells BEFORE 2D / 3D calculations.

    Same random seed makes the 2D and 3D runs
    use the same sampled cell IDs,
    assuming the input file is unchanged.
    """

    cell_ids = (
        nuclear_tx["cell"]
        .dropna()
        .astype(str)
        .unique()
    )

    total_cells = len(cell_ids)

    if (
        sample_size is None
        or sample_size <= 0
        or sample_size >= total_cells
    ):

        print(
            f"  Sampling skipped: "
            f"using all {total_cells:,} cells."
        )

        return nuclear_tx.copy()

    rng = np.random.RandomState(
        random_state
    )

    sampled_cells = rng.choice(
        cell_ids,
        size=sample_size,
        replace=False
    )

    sampled_tx = nuclear_tx[
        nuclear_tx["cell"].isin(
            sampled_cells
        )
    ].copy()

    print(
        f"  Sampled cells: "
        f"{len(sampled_cells):,} / "
        f"{total_cells:,}"
    )

    print(
        f"  Nuclear transcripts after sampling: "
        f"{len(sampled_tx):,}"
    )

    return sampled_tx


# ============================================================
# STEP 1A
# COMPUTE 2D r_norm
# ============================================================

def compute_simple_rnorm(
    nuclear_tx: pd.DataFrame
) -> pd.DataFrame:

    print(
        "\nStep 1A: Computing 2D r_norm..."
    )

    nuclear_tx = nuclear_tx.copy()

    grouped = nuclear_tx.groupby(
        "cell"
    )

    centers = (
        grouped[["x", "y"]]
        .mean()
        .rename(
            columns={
                "x": "cx",
                "y": "cy",
            }
        )
    )

    def calc_radius(sub):

        cx = sub["x"].mean()
        cy = sub["y"].mean()

        dist = np.sqrt(
            (sub["x"] - cx) ** 2
            +
            (sub["y"] - cy) ** 2
        )

        return float(
            dist.max()
        )

    centers["radius_2d"] = (
        grouped
        .apply(
            calc_radius
        )
    )

    def add_rnorm_2d(sub):

        cell_id = sub.name

        radius = float(
            centers.loc[
                cell_id,
                "radius_2d"
            ]
        )

        cx = float(
            centers.loc[
                cell_id,
                "cx"
            ]
        )

        cy = float(
            centers.loc[
                cell_id,
                "cy"
            ]
        )

        dist = np.sqrt(
            (sub["x"] - cx) ** 2
            +
            (sub["y"] - cy) ** 2
        )

        sub = sub.copy()

        sub["r_norm_2d"] = (
            dist
            /
            (radius + 1e-12)
        )

        return sub

    out = (
        nuclear_tx
        .groupby(
            "cell",
            group_keys=False
        )
        .apply(
            add_rnorm_2d
        )
        .copy()
    )

    out["r_norm_2d"] = (
        out["r_norm_2d"]
        .clip(
            0.0,
            1.0
        )
    )

    print(
        "  2D r_norm completed."
    )

    return out


# ============================================================
# STEP 1B
# NUCLEAR DIAMETERS
# ============================================================

def compute_nuclear_diameters(
    tx_df: pd.DataFrame
) -> pd.DataFrame:

    print(
        "\nStep 1B: Computing nuclear diameters..."
    )

    results = []

    grouped = tx_df.groupby(
        "cell"
    )

    for cell, group in tqdm(
        grouped,
        total=tx_df["cell"].nunique(),
        desc="Nuclear diameters"
    ):

        cx = group["x"].mean()
        cy = group["y"].mean()

        dists = np.sqrt(
            (group["x"] - cx) ** 2
            +
            (group["y"] - cy) ** 2
        )

        # Original pipeline definition
        diameter = float(
            dists.max()
        )

        results.append(
            {
                "cell": str(cell),
                "diameter_px": diameter,
                "n_transcripts": int(
                    len(group)
                ),
            }
        )

    diam_df = pd.DataFrame(
        results
    )

    print(
        f"  Diameters computed for "
        f"{len(diam_df):,} cells."
    )

    return diam_df


def print_statistics(
    diam_df: pd.DataFrame
) -> None:

    print(
        "\n========== Nuclear diameter statistics =========="
    )

    print(
        diam_df["diameter_px"].describe(
            percentiles=[
                0.05,
                0.10,
                0.25,
                0.50,
                0.75,
                0.90,
                0.95,
            ]
        )
    )

    print(
        f"\nTotal cells: "
        f"{len(diam_df):,}"
    )

    print(
        f"Total nuclear transcripts: "
        f"{diam_df['n_transcripts'].sum():,}"
    )

    print(
        "=================================================="
    )


def plot_diameter_distribution(
    diam_df: pd.DataFrame,
    out_dir: str
) -> None:

    if len(diam_df) == 0:
        return

    sns.set_style(
        "whitegrid"
    )

    fig, axes = plt.subplots(
        2,
        2,
        figsize=(12, 10)
    )

    # Histogram
    sns.histplot(
        diam_df["diameter_px"],
        bins=50,
        kde=True,
        ax=axes[0, 0]
    )

    axes[0, 0].set_xlabel(
        "Diameter (pixels)"
    )

    axes[0, 0].set_ylabel(
        "Number of cells"
    )

    axes[0, 0].set_title(
        "Distribution of nuclear diameters"
    )

    # Boxplot
    sns.boxplot(
        y=diam_df["diameter_px"],
        ax=axes[0, 1]
    )

    axes[0, 1].set_ylabel(
        "Diameter (pixels)"
    )

    axes[0, 1].set_title(
        "Boxplot of diameters"
    )

    # Scatter
    axes[1, 0].scatter(
        diam_df["n_transcripts"],
        diam_df["diameter_px"],
        alpha=0.5,
        s=10
    )

    axes[1, 0].set_xlabel(
        "Number of nuclear transcripts"
    )

    axes[1, 0].set_ylabel(
        "Diameter (pixels)"
    )

    axes[1, 0].set_title(
        "Diameter vs transcript count"
    )

    if len(diam_df) >= 2:

        slope, intercept, r_value, _, _ = linregress(
            diam_df["n_transcripts"],
            diam_df["diameter_px"]
        )

        x_fit = np.linspace(
            diam_df["n_transcripts"].min(),
            diam_df["n_transcripts"].max(),
            100
        )

        y_fit = (
            slope * x_fit
            +
            intercept
        )

        axes[1, 0].plot(
            x_fit,
            y_fit,
            "--",
            label=f"R²={r_value**2:.2f}"
        )

        axes[1, 0].legend()

    # CDF
    sorted_diam = np.sort(
        diam_df["diameter_px"]
    )

    cdf = (
        np.arange(
            1,
            len(sorted_diam) + 1
        )
        /
        len(sorted_diam)
    )

    axes[1, 1].plot(
        sorted_diam,
        cdf,
        "."
    )

    axes[1, 1].set_xlabel(
        "Diameter (pixels)"
    )

    axes[1, 1].set_ylabel(
        "Cumulative fraction"
    )

    axes[1, 1].set_title(
        "Empirical CDF"
    )

    plt.tight_layout()

    out_fig = os.path.join(
        out_dir,
        "nuclear_diameter_analysis.png"
    )

    plt.savefig(
        out_fig,
        dpi=300
    )

    plt.close()

    print(
        f"  Diameter plot saved:\n"
        f"  {out_fig}"
    )


# ============================================================
# STEP 1C
# WICKSELL 3D CORRECTION
# ============================================================

def wicksell_invert_cell_radius(
    cell_df: pd.DataFrame,
    diameter_col: str = DIAMETER_COL,
    max_diameter: float = None,
    sample_size: int = 40000,
    random_state: int = 42
) -> pd.DataFrame:

    print(
        "\nStep 1C: Starting Wicksell inversion..."
    )

    cell_df = cell_df.copy()

    cell_df[DIAM_CELL_COL] = (
        cell_df[DIAM_CELL_COL]
        .astype(str)
    )

    cell_df[diameter_col] = pd.to_numeric(
        cell_df[diameter_col],
        errors="coerce"
    )

    cell_df = cell_df.dropna(
        subset=[
            diameter_col
        ]
    ).copy()

    print(
        f"  Original cells: "
        f"{len(cell_df):,}"
    )

    if max_diameter is not None:

        cell_df = cell_df[
            cell_df[diameter_col]
            <=
            float(max_diameter)
        ].copy()

        print(
            f"  Cells after diameter filter: "
            f"{len(cell_df):,}"
        )

    # observed section radius
    cell_df["r_obs"] = (
        cell_df[diameter_col]
        /
        2.0
    )

    if len(cell_df) > sample_size:

        cell_df = cell_df.sample(
            n=sample_size,
            random_state=random_state
        ).copy()

        print(
            f"  Wicksell sampling: "
            f"{sample_size:,} cells"
        )

    r_obs = cell_df[
        "r_obs"
    ].to_numpy(
        dtype=float
    )

    if len(r_obs) < 10:

        raise RuntimeError(
            "Too few cells for Wicksell inversion."
        )

    # --------------------------------------------------------
    # KDE
    # --------------------------------------------------------

    print(
        "  Wicksell 1/3: KDE"
    )

    kde = gaussian_kde(
        r_obs,
        bw_method="scott"
    )

    r_grid = np.linspace(
        max(
            r_obs.min(),
            1e-6
        ),
        r_obs.max(),
        300
    )

    g_r = kde(
        r_grid
    )

    # --------------------------------------------------------
    # Abel integral
    # --------------------------------------------------------

    print(
        "  Wicksell 2/3: Abel integral"
    )

    R_grid = np.linspace(
        max(
            r_obs.min(),
            1e-6
        ),
        r_obs.max() * 1.5,
        350
    )

    F_R = np.zeros_like(
        R_grid
    )

    for i, R in enumerate(
        tqdm(
            R_grid,
            desc="Abel integral"
        )
    ):

        mask = (
            r_grid <= R
        )

        r_vals = r_grid[
            mask
        ]

        g_vals = g_r[
            mask
        ]

        if len(r_vals) < 2:
            continue

        integrand = (
            g_vals
            /
            np.sqrt(
                np.maximum(
                    R**2
                    -
                    r_vals**2,
                    1e-8
                )
            )
        )

        F_R[i] = np.trapz(
            integrand,
            r_vals
        )

    # --------------------------------------------------------
    # Recover 3D radius distribution
    # --------------------------------------------------------

    print(
        "  Wicksell 3/3: Recovering f(R)"
    )

    dF = np.gradient(
        F_R,
        R_grid
    )

    f_R = (
        -2
        *
        R_grid
        /
        np.pi
        *
        dF
    )

    f_R[
        f_R < 0
    ] = 0.0

    f_R = gaussian_filter1d(
        f_R,
        sigma=2
    )

    total_area = np.trapz(
        f_R,
        R_grid
    )

    f_R /= (
        total_area
        +
        1e-12
    )

    mu_R = np.trapz(
        R_grid
        *
        f_R,
        R_grid
    )

    sigma_R = np.sqrt(
        np.trapz(
            (
                R_grid
                -
                mu_R
            ) ** 2
            *
            f_R,
            R_grid
        )
    )

    print(
        f"  Wicksell mean R: "
        f"{mu_R:.3f}"
    )

    print(
        f"  Wicksell SD R: "
        f"{sigma_R:.3f}"
    )

    # --------------------------------------------------------
    # Estimate R for each observed section
    # --------------------------------------------------------

    print(
        "  Estimating true radius for each cell..."
    )

    r_lookup = r_grid

    R_lookup = np.zeros_like(
        r_lookup
    )

    for i, r in enumerate(
        tqdm(
            r_lookup,
            desc="Radius lookup"
        )
    ):

        mask = (
            R_grid >= r
        )

        if np.sum(mask) < 5:

            R_lookup[i] = r

            continue

        weights = f_R[
            mask
        ]

        R_vals = R_grid[
            mask
        ]

        R_lookup[i] = (
            np.sum(
                R_vals
                *
                weights
            )
            /
            (
                np.sum(weights)
                +
                1e-12
            )
        )

    interp_func = interp1d(
        r_lookup,
        R_lookup,
        fill_value="extrapolate",
        bounds_error=False
    )

    cell_df["R_est"] = interp_func(
        cell_df["r_obs"].values
    )

    cell_df["R_est"] = np.maximum(
        cell_df["R_est"],
        cell_df["r_obs"]
    )

    print(
        f"  R_est range: "
        f"{cell_df['R_est'].min():.3f} - "
        f"{cell_df['R_est'].max():.3f}"
    )

    return cell_df


def apply_3d_correction(
    tx_df: pd.DataFrame,
    cell_df: pd.DataFrame
) -> pd.DataFrame:

    print(
        "\nStep 1D: Applying 3D correction..."
    )

    tx_df = tx_df.copy()
    cell_df = cell_df.copy()

    tx_df["cell"] = (
        tx_df["cell"]
        .astype(str)
    )

    cell_df["cell"] = (
        cell_df["cell"]
        .astype(str)
    )

    tx_df = tx_df.merge(
        cell_df[
            [
                "cell",
                "r_obs",
                "R_est",
            ]
        ],
        on="cell",
        how="inner"
    )

    ratio = (
        tx_df["r_obs"].astype(float)
        /
        (
            tx_df["R_est"].astype(float)
            +
            1e-12
        )
    )

    r2d = (
        tx_df["r_norm_2d"]
        .astype(float)
        .clip(
            0.0,
            1.0
        )
    )

    val = (
        1.0
        -
        ratio**2
        +
        (
            r2d
            *
            ratio
        ) ** 2
    )

    val = np.maximum(
        val,
        0.0
    )

    tx_df["r_norm_3d"] = (
        np.sqrt(
            val
        )
        .clip(
            upper=1.0
        )
    )

    return tx_df


# ============================================================
# STEP 1 MAIN
# ============================================================

def step1_compute_rnorm(
    tx_file: str,
    out_dir: str,
    mode: str
) -> str:

    tx_df = load_transcripts(
        tx_file
    )

    nuclear_tx = get_nuclear_transcripts(
        tx_df
    )

    sampled_nuclear_tx = sample_cells_first(
        nuclear_tx,
        sample_size=SAMPLE_SIZE,
        random_state=RANDOM_STATE
    )

    # Always calculate 2D first
    sampled_nuclear_tx = compute_simple_rnorm(
        sampled_nuclear_tx
    )

    mode = mode.upper().strip()

    # ========================================================
    # 2D MODE
    # ========================================================

    if mode == "2D":

        print(
            "\nSelected mode: 2D"
        )

        sampled_nuclear_tx[
            "r_norm"
        ] = sampled_nuclear_tx[
            "r_norm_2d"
        ]

        out_file = os.path.join(
            out_dir,
            "tx_nuclear_with_rnorm.csv"
        )

        sampled_nuclear_tx.to_csv(
            out_file,
            index=False
        )

        print(
            "\n2D r_norm file saved:"
        )

        print(
            f"  {out_file}"
        )

        return out_file

    # ========================================================
    # 3D MODE
    # ========================================================

    print(
        "\nSelected mode: 3D"
    )

    diam_df = compute_nuclear_diameters(
        sampled_nuclear_tx
    )

    diam_df.to_csv(
        DIAMETER_CSV_OUT,
        index=False
    )

    print(
        f"  Diameter table saved:\n"
        f"  {DIAMETER_CSV_OUT}"
    )

    print_statistics(
        diam_df
    )

    plot_diameter_distribution(
        diam_df,
        OUT_DESC
    )

    cell_df = wicksell_invert_cell_radius(
        diam_df,
        diameter_col=DIAMETER_COL,
        max_diameter=MAX_DIAMETER,
        sample_size=40000,
        random_state=RANDOM_STATE
    )

    radius_out = os.path.join(
        OUT_PART1,
        "cells_with_estimated_radius_sampled.csv"
    )

    cell_df.to_csv(
        radius_out,
        index=False
    )

    print(
        f"  Estimated radius table saved:\n"
        f"  {radius_out}"
    )

    tx_3d = apply_3d_correction(
        sampled_nuclear_tx,
        cell_df
    )

    # This is THE selected r_norm
    tx_3d["r_norm"] = (
        tx_3d["r_norm_3d"]
    )

    out_file = os.path.join(
        out_dir,
        "tx_nuclear_with_rnorm.csv"
    )

    tx_3d.to_csv(
        out_file,
        index=False
    )

    print(
        "\n3D r_norm file saved:"
    )

    print(
        f"  {out_file}"
    )

    # --------------------------------------------------------
    # 2D vs 3D diagnostic
    # --------------------------------------------------------

    print(
        "\nComparing 2D and 3D r_norm..."
    )

    r2d = tx_3d[
        "r_norm_2d"
    ].values

    r3d = tx_3d[
        "r_norm_3d"
    ].values

    pearson_r, _ = pearsonr(
        r2d,
        r3d
    )

    spearman_r, _ = spearmanr(
        r2d,
        r3d
    )

    print(
        f"  Pearson r: "
        f"{pearson_r:.4f}"
    )

    print(
        f"  Spearman r: "
        f"{spearman_r:.4f}"
    )

    max_points = 50000

    if len(r2d) > max_points:

        rng = np.random.RandomState(
            RANDOM_STATE
        )

        idx = rng.choice(
            len(r2d),
            max_points,
            replace=False
        )

        x_plot = r2d[
            idx
        ]

        y_plot = r3d[
            idx
        ]

    else:

        x_plot = r2d
        y_plot = r3d

    plt.figure(
        figsize=(8, 8)
    )

    plt.scatter(
        x_plot,
        y_plot,
        s=12,
        alpha=0.02
    )

    plt.plot(
        [0, 1],
        [0, 1],
        "--",
        linewidth=1
    )

    plt.xlabel(
        "2D r_norm"
    )

    plt.ylabel(
        "3D r_norm"
    )

    plt.title(
        "2D vs 3D r_norm\n"
        f"Pearson={pearson_r:.3f}, "
        f"Spearman={spearman_r:.3f}"
    )

    plt.xlim(
        0,
        1
    )

    plt.ylim(
        0,
        1
    )

    plt.tight_layout()

    plot_file = os.path.join(
        OUT_PART1,
        "r_norm_2D_vs_3D_scatter.png"
    )

    plt.savefig(
        plot_file,
        dpi=300
    )

    plt.close()

    print(
        f"  Comparison plot saved:\n"
        f"  {plot_file}"
    )

    return out_file


# ============================================================
# STEP 2
# PDE MODEL
# ============================================================

def normalize_r(
    r_norm,
    mode="center0_edge1"
):

    r = np.asarray(
        r_norm,
        dtype=float
    )

    if mode == "edge0_center1":
        r = 1.0 - r

    return np.clip(
        r,
        0.0,
        1.0
    )


def pde_density_r(
    r,
    lmbda,
    beta
):

    r = np.asarray(
        r,
        dtype=float
    )

    I0b = i0(
        1.0 / lmbda
    )

    I1b = i1(
        1.0 / lmbda
    )

    denom = (
        beta * I0b
        +
        I1b / lmbda
    )

    C1 = (
        -beta
        /
        denom
    )

    c_r = (
        1.0
        +
        C1
        *
        i0(
            r / lmbda
        )
    )

    return np.maximum(
        r * c_r,
        1e-300
    )


def neg_loglik_pde(
    params,
    r
):

    a, b = params

    lmbda = (
        np.log1p(
            np.exp(a)
        )
        +
        1e-6
    )

    beta = (
        np.log1p(
            np.exp(b)
        )
        +
        1e-6
    )

    grid = np.linspace(
        0.0,
        1.0,
        1001
    )

    f_grid = pde_density_r(
        grid,
        lmbda,
        beta
    )

    Z = np.trapz(
        f_grid,
        grid
    )

    f_r = (
        pde_density_r(
            r,
            lmbda,
            beta
        )
        /
        (
            Z
            +
            1e-12
        )
    )

    ll = np.sum(
        np.log(
            f_r
        )
    )

    reg = (
        1e-4
        *
        (
            a * a
            +
            b * b
        )
    )

    return -(
        ll - reg
    )


def fit_cell_pde(
    cell,
    r_values,
    min_tx=50,
    r_mode="center0_edge1"
):

    if len(r_values) < min_tx:
        return None

    r = normalize_r(
        r_values,
        mode=r_mode
    )

    if np.std(r) < 1e-6:
        return None

    x0 = np.array(
        [
            np.log(
                np.expm1(
                    0.3
                )
            ),
            np.log(
                np.expm1(
                    1.0
                )
            )
        ]
    )

    res = minimize(
        neg_loglik_pde,
        x0,
        args=(r,),
        method="L-BFGS-B"
    )

    if not res.success:
        return None

    a, b = res.x

    lmbda = (
        np.log1p(
            np.exp(a)
        )
        +
        1e-6
    )

    beta = (
        np.log1p(
            np.exp(b)
        )
        +
        1e-6
    )

    grid = np.linspace(
        0.0,
        1.0,
        2001
    )

    f_grid = pde_density_r(
        grid,
        lmbda,
        beta
    )

    Z = np.trapz(
        f_grid,
        grid
    )

    pdf = (
        f_grid
        /
        (
            Z
            +
            1e-12
        )
    )

    cdf = (
        np.cumsum(
            pdf
        )
        *
        (
            grid[1]
            -
            grid[0]
        )
    )

    def model_cdf(x):

        x_arr = np.asarray(
            x,
            dtype=float
        )

        idx = np.searchsorted(
            grid,
            x_arr,
            side="right"
        )

        idx = np.clip(
            idx,
            0,
            len(cdf) - 1
        )

        return cdf[
            idx
        ]

    ks_stat = kstest(
        r,
        model_cdf
    ).statistic

    ll = -neg_loglik_pde(
        res.x,
        r
    )

    aic = (
        2 * 2
        -
        2 * ll
    )

    return {
        "cell": str(cell),
        "lambda": float(lmbda),
        "beta": float(beta),
        "loglik": float(ll),
        "AIC": float(aic),
        "KS": float(ks_stat),
        "n_tx": int(len(r)),
        "mean_r": float(np.mean(r)),
        "median_r": float(np.median(r)),
        "norm_mode": NORM_MODE,
    }


def step2_fit_all(
    tx_rnorm_file,
    out_dir,
    min_tx=50
):

    print(
        "\nStep 2: Fitting PDE per cell..."
    )

    print(
        f"  Mode = {NORM_MODE}"
    )

    df = pd.read_csv(
        tx_rnorm_file,
        low_memory=False
    )

    if (
        "cell" not in df.columns
        or
        "r_norm" not in df.columns
    ):

        raise RuntimeError(
            "Input file missing "
            "cell or r_norm."
        )

    # IMPORTANT:
    # r_norm is already selected according to 2D / 3D.
    cell_groups = (
        df.groupby(
            "cell"
        )["r_norm"]
        .apply(list)
    )

    print(
        f"  Cells found: "
        f"{len(cell_groups):,}"
    )

    # --------------------------------------------------------
    # Example cell
    # --------------------------------------------------------

    rng = np.random.RandomState(
        RANDOM_STATE
    )

    rand_cell = rng.choice(
        cell_groups.index,
        1
    )[0]

    example_res = fit_cell_pde(
        rand_cell,
        cell_groups[rand_cell],
        min_tx=min_tx
    )

    if example_res:

        r = normalize_r(
            cell_groups[rand_cell]
        )

        lmbda = example_res[
            "lambda"
        ]

        beta = example_res[
            "beta"
        ]

        grid = np.linspace(
            0,
            1,
            200
        )

        f_grid = pde_density_r(
            grid,
            lmbda,
            beta
        )

        f_grid /= (
            np.trapz(
                f_grid,
                grid
            )
            +
            1e-12
        )

        plt.figure(
            figsize=(6, 4)
        )

        plt.hist(
            r,
            bins=20,
            range=(0, 1),
            density=True,
            alpha=0.5,
            label="Empirical"
        )

        plt.plot(
            grid,
            f_grid,
            "-",
            lw=2,
            label="PDE fit"
        )

        plt.xlabel(
            f"r_norm ({NORM_MODE})"
        )

        plt.ylabel(
            "Density"
        )

        plt.title(
            f"{NORM_MODE} mode\n"
            f"Example Cell {rand_cell}\n"
            f"λ={lmbda:.3f}, "
            f"β={beta:.3f}, "
            f"KS={example_res['KS']:.3f}"
        )

        plt.legend()

        plt.tight_layout()

        example_fig = os.path.join(
            out_dir,
            f"fit_example_{rand_cell}.png"
        )

        plt.savefig(
            example_fig,
            dpi=300
        )

        plt.close()

        print(
            f"  Example fit saved:\n"
            f"  {example_fig}"
        )

    # --------------------------------------------------------
    # Fit all cells
    # --------------------------------------------------------

    results = []

    total = len(
        cell_groups.index
    )

    for i, cell in enumerate(
        cell_groups.index,
        start=1
    ):

        res = fit_cell_pde(
            cell,
            cell_groups[cell],
            min_tx=min_tx
        )

        if res:
            results.append(
                res
            )

        if (
            i % 500 == 0
            or
            i == total
        ):

            sys.stdout.write(
                f"\r  Progress: "
                f"{i:,}/{total:,} "
                f"({i/total*100:.1f}%)"
            )

            sys.stdout.flush()

    print(
        "\n  PDE fitting finished."
    )

    results_df = pd.DataFrame(
        results
    )

    out_csv = os.path.join(
        out_dir,
        "fit_results_all_cells.csv"
    )

    results_df.to_csv(
        out_csv,
        index=False
    )

    print(
        f"  Fit results saved:\n"
        f"  {out_csv}"
    )

    # --------------------------------------------------------
    # Parameter distributions
    # --------------------------------------------------------

    if len(results_df) > 0:

        plt.figure(
            figsize=(12, 4)
        )

        plt.subplot(
            1,
            3,
            1
        )

        plt.hist(
            results_df["lambda"],
            bins=50
        )

        plt.xlabel(
            "lambda"
        )

        plt.title(
            f"{NORM_MODE}: λ"
        )

        plt.subplot(
            1,
            3,
            2
        )

        plt.hist(
            results_df["beta"],
            bins=50
        )

        plt.xlabel(
            "beta"
        )

        plt.title(
            f"{NORM_MODE}: β"
        )

        plt.subplot(
            1,
            3,
            3
        )

        plt.hist(
            results_df["KS"],
            bins=50
        )

        plt.xlabel(
            "KS"
        )

        plt.title(
            f"{NORM_MODE}: KS"
        )

        plt.tight_layout()

        fig_out = os.path.join(
            out_dir,
            "fit_params_distribution.png"
        )

        plt.savefig(
            fig_out,
            dpi=300
        )

        plt.close()

        print(
            f"  Parameter plot saved:\n"
            f"  {fig_out}"
        )

    return out_csv


# ============================================================
# STEP 3
# SPATIAL KS MAP
# ============================================================

def step3_plot_ks_map(
    fit_csv,
    meta_file,
    out_dir,
    vmin_percentile=PLOT_VMIN_PERCENTILE,
    vmax_percentile=PLOT_VMAX_PERCENTILE
):

    if (
        meta_file is None
        or
        not os.path.exists(
            meta_file
        )
    ):

        print(
            "\nStep 3: Metadata not found. "
            "Skipping KS map."
        )

        return None

    print(
        "\nStep 3: Plotting spatial KS map..."
    )

    fit_df = pd.read_csv(
        fit_csv
    )

    meta_df = pd.read_csv(
        meta_file,
        low_memory=False
    )

    fit_df["cell"] = (
        fit_df["cell"]
        .astype(str)
    )

    meta_df = standardize_metadata(
        meta_df
    )

    merged = pd.merge(
        fit_df[
            [
                "cell",
                "KS",
            ]
        ],
        meta_df[
            [
                "cell",
                "meta_x",
                "meta_y",
            ]
        ],
        on="cell",
        how="inner"
    )

    if merged.empty:

        print(
            "  No matching cell IDs. "
            "Skipping."
        )

        return None

    vmin, vmax = get_percentile_limits(
        merged["KS"],
        low_pct=vmin_percentile,
        high_pct=vmax_percentile
    )

    merged[
        "KS_clipped"
    ] = merged[
        "KS"
    ].clip(
        vmin,
        vmax
    )

    plt.figure(
        figsize=(10, 10)
    )

    sc = plt.scatter(
        merged["meta_x"],
        merged["meta_y"],
        c=merged["KS_clipped"],
        cmap="YlOrRd",
        s=20,
        edgecolors="none",
        vmin=vmin,
        vmax=vmax
    )

    cbar = plt.colorbar(
        sc,
        fraction=0.046,
        pad=0.04
    )

    cbar.set_label(
        "KS"
    )

    plt.xlabel(
        "X coordinate"
    )

    plt.ylabel(
        "Y coordinate"
    )

    plt.title(
        f"Spatial KS Map ({NORM_MODE})"
    )

    plt.gca().invert_yaxis()

    out_file = os.path.join(
        out_dir,
        "spatial_KS_map.png"
    )

    plt.savefig(
        out_file,
        dpi=600,
        bbox_inches="tight"
    )

    plt.close()

    print(
        f"  KS map saved:\n"
        f"  {out_file}"
    )

    return out_file


# ============================================================
# STEP 4
# EXPORT PROBABILITY
# ============================================================

def s_flux_r(
    r,
    lmbda,
    beta
):

    I0b = i0(
        1.0 / lmbda
    )

    I1b = i1(
        1.0 / lmbda
    )

    C1 = (
        -beta
        /
        (
            beta * I0b
            +
            I1b / lmbda
        )
    )

    c_r = (
        1.0
        +
        C1
        *
        i0(
            r / lmbda
        )
    )

    s_r = np.maximum(
        r * c_r,
        1e-10
    )

    return s_r


def step4_compute_delta_and_future(
    tx_rnorm_file,
    fit_csv,
    out_dir
):

    print(
        "\nStep 4: Computing export probabilities..."
    )

    print(
        f"  Mode = {NORM_MODE}"
    )

    tx_df = pd.read_csv(
        tx_rnorm_file,
        low_memory=False
    )

    fit_df = pd.read_csv(
        fit_csv,
        low_memory=False
    )

    tx_df = standardize_transcripts(
        tx_df,
        require_gene=True
    )

    tx_df["cell"] = (
        tx_df["cell"]
        .astype(str)
    )

    fit_df["cell"] = (
        fit_df["cell"]
        .astype(str)
    )

    if "r_norm" not in tx_df.columns:

        raise RuntimeError(
            "tx_rnorm_file does not contain "
            "selected r_norm column."
        )

    all_results = []

    fit_cells = set(
        fit_df["cell"].unique()
    )

    fit_lookup = (
        fit_df
        .set_index(
            "cell"
        )
    )

    grouped = tx_df.groupby(
        "cell"
    )

    total_cells = tx_df[
        "cell"
    ].nunique()

    for cell, group in tqdm(
        grouped,
        total=total_cells,
        desc=f"Export probability ({NORM_MODE})"
    ):

        if cell not in fit_cells:
            continue

        row = fit_lookup.loc[
            cell
        ]

        lmbda = float(
            row["lambda"]
        )

        beta = float(
            row["beta"]
        )

        # ====================================================
        # CRITICAL FIX
        #
        # Use selected r_norm.
        #
        # 2D:
        #     r_norm = r_norm_2d
        #
        # 3D:
        #     r_norm = r_norm_3d
        # ====================================================

        rvals = normalize_r(
            group[
                "r_norm"
            ].values
        )

        s_vals = s_flux_r(
            rvals,
            lmbda,
            beta
        )

        alpha = (
            beta
            *
            len(rvals)
            /
            (
                np.sum(
                    s_vals
                )
                +
                1e-10
            )
        )

        p_export = (
            1
            -
            np.exp(
                -alpha
                *
                s_vals
            )
        )

        tmp = pd.DataFrame(
            {
                "cell": (
                    [cell]
                    *
                    len(
                        p_export
                    )
                ),
                "gene": group[
                    "gene"
                ].values,
                "p_export": p_export,
            }
        )

        all_results.append(
            tmp
        )

    if len(all_results) == 0:

        raise RuntimeError(
            "No export probabilities computed."
        )

    all_df = pd.concat(
        all_results,
        axis=0,
        ignore_index=True
    )

    print(
        f"  Per-transcript export entries: "
        f"{len(all_df):,}"
    )

    # --------------------------------------------------------
    # Gene-cell export probability
    # --------------------------------------------------------

    delta_export_prob = (
        all_df
        .groupby(
            [
                "cell",
                "gene"
            ]
        )[
            "p_export"
        ]
        .sum()
        .reset_index()
    )

    # Faster than iterative DataFrame assignment
    delta_mat = (
        delta_export_prob
        .pivot(
            index="cell",
            columns="gene",
            values="p_export"
        )
        .fillna(
            0.0
        )
    )

    delta_mat.index = (
        delta_mat.index
        .astype(str)
    )

    # --------------------------------------------------------
    # Save raw delta
    # --------------------------------------------------------

    delta_out = os.path.join(
        out_dir,
        "delta_export_prob.csv"
    )

    delta_mat.to_csv(
        delta_out
    )

    print(
        f"  Raw delta saved:\n"
        f"  {delta_out}"
    )

    # --------------------------------------------------------
    # Median standardized delta
    # --------------------------------------------------------

    delta_median = (
        delta_mat
        .median(
            axis=0
        )
    )

    delta_standardized = (
        delta_mat
        .subtract(
            delta_median,
            axis=1
        )
    )

    delta_med_out = os.path.join(
        out_dir,
        "delta_export_prob_median.csv"
    )

    delta_standardized.to_csv(
        delta_med_out
    )

    print(
        f"  Median-standardized delta saved:\n"
        f"  {delta_med_out}"
    )

    # --------------------------------------------------------
    # Cytoplasm matrix
    # --------------------------------------------------------

    print(
        "\n  Loading complete transcript file "
        "for cytoplasm counts..."
    )

    all_tx = pd.read_csv(
        TX_IN_FILE,
        low_memory=False
    )

    all_tx = standardize_transcripts(
        all_tx,
        require_gene=True
    )

    df_cyto_trans = all_tx[
        all_tx[
            "CellComp"
        ]
        .astype(str)
        .str.lower()
        ==
        CYTOPLASM_LABEL.lower()
    ].copy()

    cyto_mat = pd.crosstab(
        df_cyto_trans[
            "cell"
        ],
        df_cyto_trans[
            "gene"
        ]
    )

    cyto_mat.index = (
        cyto_mat.index
        .astype(str)
    )

    print(
        f"  Cytoplasm matrix shape: "
        f"{cyto_mat.shape}"
    )

    # --------------------------------------------------------
    # Align cytoplasm and delta
    # --------------------------------------------------------

    common_cells = (
        cyto_mat.index
        .intersection(
            delta_mat.index
        )
    )

    common_genes = (
        cyto_mat.columns
        .intersection(
            delta_mat.columns
        )
    )

    print(
        f"  Common cells: "
        f"{len(common_cells):,}"
    )

    print(
        f"  Common genes: "
        f"{len(common_genes):,}"
    )

    cyto_mat_common = (
        cyto_mat
        .loc[
            common_cells,
            common_genes
        ]
        .copy()
    )

    delta_common = (
        delta_mat
        .loc[
            common_cells,
            common_genes
        ]
        .copy()
    )

    # ========================================================
    # IMPORTANT:
    # Cytoplasm matrix is now saved inside mode_2D or mode_3D
    # ========================================================

    cyto_out = os.path.join(
        out_dir,
        "transcripts_cytoplasm_matrix_common.csv"
    )

    cyto_mat_common.to_csv(
        cyto_out
    )

    print(
        f"  Cytoplasm matrix saved:\n"
        f"  {cyto_out}"
    )

    # --------------------------------------------------------
    # Future matrix
    # --------------------------------------------------------

    future_matrix = (
        cyto_mat_common
        .astype(float)
        +
        delta_common
        .astype(float)
    )

    future_out = os.path.join(
        out_dir,
        "future.csv"
    )

    future_matrix.to_csv(
        future_out
    )

    print(
        f"  Future matrix saved:\n"
        f"  {future_out}"
    )

    return (
        delta_out,
        delta_med_out,
        cyto_out,
        future_out
    )


# ============================================================
# MAIN PIPELINE
# ============================================================

def run_pipeline():

    print(
        "\n"
        "############################################################"
    )

    print(
        "# Spatial-Velo nuclear export pipeline"
    )

    print(
        f"# NORMALIZATION MODE: {NORM_MODE}"
    )

    print(
        f"# OUTPUT DIRECTORY:"
    )

    print(
        f"# {OUT_DIR}"
    )

    print(
        "############################################################"
        "\n"
    )

    # --------------------------------------------------------
    # Step 1
    # --------------------------------------------------------

    tx_rnorm_file = step1_compute_rnorm(
        TX_IN_FILE,
        OUT_PART1,
        mode=NORM_MODE
    )

    # --------------------------------------------------------
    # Step 2
    # --------------------------------------------------------

    fit_csv = step2_fit_all(
        tx_rnorm_file,
        OUT_PART2,
        min_tx=MIN_TX_PER_CELL
    )

    # --------------------------------------------------------
    # Step 3
    # --------------------------------------------------------

    ks_map = None

    try:

        ks_map = step3_plot_ks_map(
            fit_csv,
            META_FILE,
            OUT_PART2
        )

    except Exception as e:

        print(
            "\nStep 3 skipped or failed:"
        )

        print(
            e
        )

    # --------------------------------------------------------
    # Step 4
    # --------------------------------------------------------

    (
        delta_out,
        delta_med_out,
        cyto_out,
        future_out
    ) = step4_compute_delta_and_future(
        tx_rnorm_file,
        fit_csv,
        OUT_PART3
    )

    # --------------------------------------------------------
    # Final summary
    # --------------------------------------------------------

    print(
        "\n"
        "============================================================"
    )

    print(
        "PIPELINE FINISHED"
    )

    print(
        "============================================================"
    )

    print(
        f"\nMode:\n"
        f"  {NORM_MODE}"
    )

    print(
        f"\nMain output directory:\n"
        f"  {OUT_DIR}"
    )

    print(
        f"\nStep 1 - r_norm:\n"
        f"  {tx_rnorm_file}"
    )

    print(
        f"\nStep 2 - PDE fit:\n"
        f"  {fit_csv}"
    )

    print(
        f"\nStep 3 - KS map:\n"
        f"  {ks_map if ks_map else 'skipped'}"
    )

    print(
        f"\nStep 4 - delta:\n"
        f"  {delta_out}"
    )

    print(
        f"\nStep 4 - delta median:\n"
        f"  {delta_med_out}"
    )

    print(
        f"\nStep 4 - cytoplasm:\n"
        f"  {cyto_out}"
    )

    print(
        f"\nStep 4 - future:\n"
        f"  {future_out}"
    )

    print(
        "\n"
        "============================================================"
    )


if __name__ == "__main__":
    run_pipeline()